# qbraid.runtime.qudora

> **Billing notice:** the QUDORA emulators are currently **free to run** (0 credits). You need a qBraid API key, but no QUDORA account.

[QUDORA](https://qudora.com) builds **trapped-ion** quantum computers. qBraid gives you two of
its cloud emulators, both noisy statevector simulators whose noise model is tuned to QUDORA hardware:

- **Qamelion** (32 qubits): QUDORA's newer emulator, with an adjustable noise model meant to
  mirror its current and upcoming trapped-ion machines.
- **QVLS-Q1 Emulator** (20 qubits): based on the noise model of the 50-qubit computer being built
  for the Quantum Valley Lower Saxony (QVLS).

Both take the same settings, so everything below works on either one.

This notebook covers:

1. Connecting and running a first circuit (which is **noisy by default**)
2. The one runtime option, `backend_settings`, and its four noise parameters
3. Switching noise off, and turning individual noise sources up
4. What happens when a setting is wrong
5. Running many circuits: no batch jobs, so a list becomes separate jobs

In [ ]:
%%capture

# "%%capture" hides the install output; comment it out if you need to debug this step.
%pip install 'qbraid[qiskit,visualization]'

## Connect to the device

`QbraidProvider` picks up your API key from `~/.qbraid/qbraidrc` or the `QBRAID_API_KEY`
environment variable. Inside qBraid Lab it is already configured for you.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()

qamelion = provider.get_device("qudora:qudora:sim:qamelion")
qvls = provider.get_device("qudora:qudora:sim:qvls-q1-emulator")

qamelion.metadata()

32 qubits, `ONLINE`, and zero pricing. The QVLS-Q1 Emulator is the same apart from its
size:

In [ ]:
for dev in (qamelion, qvls):
    meta = dev.metadata()
    print(f"{meta['name']:<20} {meta['num_qubits']:>3} qubits  {meta['status']}")

QUDORA takes OpenQASM 2 or 3, and qBraid transpiles to it for you, so a Qiskit, Cirq,
Braket, or pyQuil circuit can go straight to `run()`. We'll use Qamelion from here on.

In [ ]:
device = qamelion

device.supported_run_inputs()

## A first run

A Bell pair, written in Qiskit:

In [ ]:
from qiskit import QuantumCircuit


def ghz(num_qubits: int) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits)
    circuit.h(0)
    for qubit in range(num_qubits - 1):
        circuit.cx(qubit, qubit + 1)
    circuit.measure_all()
    return circuit


bell = ghz(2)
bell.draw()

In [ ]:
job = device.run(bell, shots=1000)
job.result().data.get_counts()

Mostly `00` and `11`, as a Bell pair should give, plus a handful of `01` and `10`, which an
ideal Bell pair never produces.

That is the emulator working as intended: **noise is on by default**. A run with no options
uses QUDORA's default noise model, so expect around 1% of shots to land on the "wrong"
outcome even on a circuit this small.

Keys are little-endian: **qubit 0 is the rightmost bit**. Flip only qubit 0 of three and
nearly every shot reads `001`:

In [ ]:
flip_q0 = QuantumCircuit(3)
flip_q0.x(0)
flip_q0.measure_all()

device.run(flip_q0, shots=200).result().data.get_counts()

## Runtime options

QUDORA's settings go in a **`runtime_options` dict**. The emulators read exactly one key,
`backend_settings`, which holds the noise model:

```python
device.run(circuit, shots=1000, runtime_options={"backend_settings": {...}})
```

| `backend_settings` key | Default | Range | What it does |
|---|---|---|---|
| `measurement_error_probability` | `0.0035` | 0 to 1 | Chance that a qubit's measurement is read out wrong |
| `single_qubit_gate_noise_strength` | `1.0` | 0 to 1 | Scales the noise on single-qubit gates. `0` makes them perfect |
| `two_qubit_gate_noise_strength` | `1.0` | 0 to 1 | Scales the noise on two-qubit gates. `0` makes them perfect |
| `dephasing_T2_time` | `60` | > 0 s, or `None` | Dephasing time T2, in seconds. `None` turns dephasing off |

You only need to give the keys you want to change; the rest keep their defaults.

Some things to know:

- Any **other key in `runtime_options`** is ignored without an error. There is no `seed`,
  for example, so two identical runs give different counts.
- An **unknown key or out-of-range value inside `backend_settings`** fails the job (see
  [When a setting is wrong](#When-a-setting-is-wrong)).

## A noiseless run

Set all four to zero (and T2 to `None`) to get an ideal statevector simulation:

In [ ]:
NOISELESS = {
    "measurement_error_probability": 0.0,
    "single_qubit_gate_noise_strength": 0.0,
    "two_qubit_gate_noise_strength": 0.0,
    "dephasing_T2_time": None,
}

ideal_job = device.run(bell, shots=1000, runtime_options={"backend_settings": NOISELESS})
ideal_job.result().data.get_counts()

Only `00` and `11` this time. This is the run to compare against when you want to know how
much of a result comes from the noise model and how much from the circuit.

## Turning one noise source up

Because unspecified keys keep their defaults, you can change one parameter and leave
the rest alone. Raise the measurement error to 20% on a circuit that should always read `1`:

In [ ]:
flip = QuantumCircuit(1)
flip.x(0)
flip.measure_all()

readout_job = device.run(
    flip,
    shots=1000,
    runtime_options={"backend_settings": {"measurement_error_probability": 0.2}},
)
readout_job.result().data.get_counts()

About a fifth of the shots now read `0`. The state was prepared correctly every time;
only the readout lies.

### Dephasing

`dephasing_T2_time` is the time over which a qubit loses its phase. The 60-second default
is very long next to a small circuit, so it barely shows. Shorten it and it takes over.

A GHZ state depends entirely on the phase between `|0000⟩` and `|1111⟩`, and the gates
that follow turn a lost phase into wrong bits. Sweep T2 with every other noise source off
and track how many shots still land on one of the two GHZ outcomes:

In [ ]:
t2_values = [1.0, 0.1, 0.03, 0.01, 0.003]
ghz4 = ghz(4)

t2_counts = []
for t2 in t2_values:
    settings = {**NOISELESS, "dephasing_T2_time": t2}
    t2_job = device.run(ghz4, shots=1000, runtime_options={"backend_settings": settings})
    t2_counts.append(t2_job.result().data.get_counts())

for t2, counts in zip(t2_values, t2_counts):
    valid = counts.get("0000", 0) + counts.get("1111", 0)
    print(f"T2 = {t2:<6} s   GHZ outcomes: {valid / 1000:6.1%}   distinct outcomes: {len(counts)}")

In [ ]:
from qbraid.visualization import plot_histogram

plot_histogram(
    [t2_counts[0], t2_counts[-1]],
    legend=[f"T2 = {t2_values[0]} s", f"T2 = {t2_values[-1]} s"],
    title="4-qubit GHZ on Qamelion, dephasing only",
)

At T2 = 1 s almost every shot is `0000` or `1111`. At 3 ms all 16 outcomes appear, and
only about one shot in five is a GHZ outcome.

## When a setting is wrong

`backend_settings` is checked by QUDORA when the job is submitted. A misspelt key or a value
outside its range fails the job straight away. When a job fails, the reason is on its
status, in `status_message`:

In [ ]:
bad_job = device.run(
    bell,
    shots=100,
    runtime_options={"backend_settings": {"measurement_error_probability": 1.5}},
)
bad_job.wait_for_final_state()

status = bad_job.status()
print(status)
print(status.status_message)

The message is only the HTTP error, `400 Client Error: Bad Request`, and doesn't name the
bad key. If you see it, check your `backend_settings` names and ranges against the table
above.

Some mistakes never reach QUDORA. A circuit wider than the device is rejected by qBraid before
anything is submitted:

In [ ]:
try:
    qvls.run(ghz(21), shots=100)
except Exception as err:
    print(f"{type(err).__name__}: {err}")

The same 21-qubit circuit fits on Qamelion, which goes up to 32. So does a circuit with no
measurements, which fails with a `status_message` asking you to add some: QUDORA would
otherwise return an empty result.

## Running many circuits

Neither QUDORA device supports qBraid **batch jobs**, so `as_batch=True` raises:

In [ ]:
import numpy as np

print("batch_job_support:", device.profile.batch_job_support)

try:
    device.run([ghz(2), ghz(3)], shots=100, as_batch=True)
except ValueError as err:
    print(f"ValueError: {err}")

Without `as_batch=True`, a list of circuits is submitted as **separate jobs**, one per
circuit, and `run()` returns a list of jobs. Each has its own ID and status, and all of
them share the `shots` and `runtime_options` you pass.

That is enough for a parameter sweep. Rotate one qubit by $\theta$ with `RY` and the
probability of measuring 1 is $\sin^2(\theta/2)$. Eight angles, eight jobs, default noise:

In [ ]:
angles = np.linspace(0, 2 * np.pi, 8)

sweep = []
for theta in angles:
    circuit = QuantumCircuit(1)
    circuit.ry(theta, 0)
    circuit.measure_all()
    sweep.append(circuit)

shots = 500
sweep_jobs = device.run(sweep, shots=shots)
sweep_counts = [sweep_job.result().data.get_counts() for sweep_job in sweep_jobs]

print(len(sweep_jobs), "jobs")
print(sweep_jobs[0].id)

In [ ]:
import matplotlib.pyplot as plt

measured = [counts.get("1", 0) / shots for counts in sweep_counts]

fine = np.linspace(0, 2 * np.pi, 200)
plt.plot(fine, np.sin(fine / 2) ** 2, label=r"$\sin^2(\theta/2)$")
plt.plot(angles, measured, "o", label="Qamelion, one job per angle")
plt.xlabel(r"$\theta$")
plt.ylabel("P(1)")
plt.legend()
plt.show()

The points follow the curve to within shot noise.

QUDORA's own API can run up to 500 programs in a single job, but the qBraid platform
submits one circuit per QUDORA job, so a list always becomes separate jobs. The jobs run
independently, and each is sampled on its own.

## Summary

- Two free emulators: `qudora:qudora:sim:qamelion` (32 qubits) and
  `qudora:qudora:sim:qvls-q1-emulator` (20 qubits). Both are trapped-ion noise models on a
  statevector simulator.
- **Noise is on by default.** Pass `runtime_options={"backend_settings": {...}}` to change it.
- `backend_settings` takes `measurement_error_probability`, `single_qubit_gate_noise_strength`,
  `two_qubit_gate_noise_strength`, and `dephasing_T2_time`. Set them all to zero (T2 to `None`) for
  an ideal run, or change just one.
- Other `runtime_options` keys are ignored, and there is no seed.
- No batch jobs: a list of circuits runs as one job per circuit.
- When a job fails, `job.status().status_message` says why.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>